# Multimodal Gait Analysis — Colab Training
**Daphnet Freezing of Gait dataset** | GPU training → Hugging Face Hub 저장

**순서:**
1. GPU 확인
2. 레포 클론 + 패키지 설치
3. Daphnet 데이터셋 다운로드
4. 학습 실행
5. HF Hub 업로드

In [ ]:
# ── Cell 1: GPU 확인 ──────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ GPU 없음 — Runtime > Change runtime type > T4 GPU 선택')

In [ ]:
# ── Cell 2: 레포 클론 ─────────────────────────────────────────────
import os

GITHUB_USER = "jg-shoealls"   # ← GitHub 유저명 확인
REPO_NAME   = "Shoealls"
BRANCH      = "claude/multimodal-walking-ai-algorithm-dUHY1"

if not os.path.exists(REPO_NAME):
    !git clone --branch {BRANCH} https://github.com/{GITHUB_USER}/{REPO_NAME}.git
else:
    print("이미 클론됨")

%cd {REPO_NAME}
!git log --oneline -3

In [ ]:
# ── Cell 3: 패키지 설치 ───────────────────────────────────────────
!pip install -q torch torchvision numpy scipy scikit-learn pyyaml matplotlib pandas huggingface_hub

In [ ]:
# ── Cell 4: Daphnet 데이터셋 다운로드 ────────────────────────────
import os, zipfile, urllib.request

DATA_DIR = "data/daphnet"
os.makedirs(DATA_DIR, exist_ok=True)

zip_path = f"{DATA_DIR}/daphnet_fog.zip"
extract_path = DATA_DIR

if not os.path.exists(f"{DATA_DIR}/dataset_fog_release"):
    print("Daphnet 데이터 다운로드 중...")
    url = "https://archive.ics.uci.edu/static/public/201/daphnet+freezing+of+gait.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    os.remove(zip_path)
    print("✅ 완료")
else:
    print("이미 존재")

# 실제 .dat 파일 위치 확인
import glob
dat_files = glob.glob(f"{DATA_DIR}/**/*.txt", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.dat", recursive=True)
print(f"데이터 파일 수: {len(dat_files)}")
if dat_files:
    print("예시:", dat_files[:3])

In [ ]:
# ── Cell 5: 데이터 경로 자동 감지 & config 업데이트 ──────────────
import glob, yaml

# .dat 또는 .txt 파일이 있는 폴더 찾기
candidates = set()
for f in glob.glob("data/daphnet/**", recursive=True):
    if os.path.isfile(f) and f.endswith(('.dat', '.txt')):
        candidates.add(os.path.dirname(f))

data_dir = sorted(candidates, key=lambda p: -len(glob.glob(f"{p}/*")))[0]
print(f"감지된 데이터 경로: {data_dir}")
print(f"파일 수: {len(os.listdir(data_dir))}")

# config 업데이트
with open("configs/daphnet.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["data_dir"] = data_dir

with open("configs/daphnet.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("✅ config 업데이트 완료")

In [ ]:
# ── Cell 6: 학습 실행 ─────────────────────────────────────────────
import sys
sys.path.insert(0, '.')

from scripts.train_daphnet import run

run(
    config_path="configs/daphnet.yaml",
    output_dir="outputs/daphnet"
)

In [ ]:
# ── Cell 7: 학습 곡선 시각화 ──────────────────────────────────────
import torch, matplotlib.pyplot as plt

ckpt = torch.load("outputs/daphnet/best_model.pt", map_location="cpu", weights_only=False)
h = ckpt["history"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(h["train_loss"], label="Train")
ax1.plot(h["val_loss"],   label="Val")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True)

ax2.plot(h["train_acc"], label="Train")
ax2.plot(h["val_acc"],   label="Val")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True)

plt.suptitle(f"Best Val Acc: {ckpt['val_accuracy']:.4f} (epoch {ckpt['epoch']})")
plt.tight_layout()
plt.savefig("outputs/daphnet/training_curves.png", dpi=150)
plt.show()
print(f"Best epoch: {ckpt['epoch']} | Val Acc: {ckpt['val_accuracy']:.4f}")

In [ ]:
# ── Cell 8: Hugging Face Hub 업로드 ──────────────────────────────
# HF_TOKEN: https://huggingface.co/settings/tokens 에서 발급
from huggingface_hub import HfApi, login
import json

HF_TOKEN    = "hf_xxxxxxxxxxxxxxxxxxxx"  # ← 여기에 토큰 입력
HF_USERNAME = "ShoeAlls"                  # ← HF 유저명
REPO_ID     = f"{HF_USERNAME}/gait-fog-classifier"

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()

# 레포 생성 (이미 있으면 무시)
api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)

# 모델 카드 생성
model_card = f"""---
license: mit
tags:
- gait-analysis
- freezing-of-gait
- parkinson
- IMU
- pytorch
datasets:
- daphnet-fog
metrics:
- accuracy
- f1
---

# Multimodal Gait Analysis — Freezing of Gait Classifier

IMU 센서 기반 보행 동결(Freezing of Gait) 이진 분류 모델.

**Dataset:** Daphnet Freezing of Gait (UCI ML Repository)  
**Best Val Accuracy:** {ckpt['val_accuracy']:.4f}  
**Best Epoch:** {ckpt['epoch']}  

## Architecture
- IMU Encoder: CNN + BiLSTM
- Pressure Encoder: CNN
- Skeleton Encoder: GCN
- Fusion: Multihead Attention Transformer
"""

with open("outputs/daphnet/README.md", "w") as f:
    f.write(model_card)

# 파일 업로드
for fname in ["best_model.pt", "training_curves.png", "README.md"]:
    fpath = f"outputs/daphnet/{fname}"
    if os.path.exists(fpath):
        api.upload_file(
            path_or_fileobj=fpath,
            path_in_repo=fname,
            repo_id=REPO_ID,
        )
        print(f"✅ 업로드: {fname}")

# config도 업로드
api.upload_file(
    path_or_fileobj="configs/daphnet.yaml",
    path_in_repo="configs/daphnet.yaml",
    repo_id=REPO_ID,
)
print(f"\n🎉 완료! 모델 페이지: https://huggingface.co/{REPO_ID}")